# M5.S1 Hybrid computing

We will follow one problem from a single CPU core to MPI processes, OpenMP threads and a GPU. The problem stays the same: a hot edge heats a cold plate.

The main work uses two C files. `heat.c` runs on one process; `heat_mpi.c` divides the plate between processes. We compile them in different ways and check that the temperatures agree before comparing times.

Work in your SSH session. At each checkpoint, stop and explain what you have observed. Keep a short record in `observations.txt`.

By the end, you should be able to explain who owns the data, which workers share memory, when data must move, and why adding workers does not always make a small problem faster.

## 1. The heat equation

Let $T(x,y,t)$ be the temperature. With constant diffusivity $\alpha$, heat diffusion is

$$\frac{\partial T}{\partial t}=\alpha\left(\frac{\partial^2T}{\partial x^2}+\frac{\partial^2T}{\partial y^2}\right).$$

We replace the plate with a square grid. The next value of an interior cell depends on its current value and its four neighbours:

$$T^{new}_{i,j}=T^{old}_{i,j}+r\left(T^{old}_{i-1,j}+T^{old}_{i+1,j}+T^{old}_{i,j-1}+T^{old}_{i,j+1}-4T^{old}_{i,j}\right),\qquad r=\frac{\alpha\Delta t}{h^2}.$$

Choose $h=1$, $\alpha=1$ and $\Delta t=0.25$. Then $r=0.25$, so the update is simply the average of the four neighbours. This is at the explicit method's stability limit for equal spacing in two dimensions. A larger time step would be unstable.

The top edge, including its corners, stays at 100 °C. The other edges and the initial interior are 0 °C. `N` includes the boundary cells. One time step uses the complete old grid to calculate a separate new grid. Updating in place would change the method.

For a first check, an interior cell directly below the hot edge must become 25 °C after one step.

### A small illustration

This Python example is only for understanding the update. HPC execution and timing use the C programs on allocated nodes.

In [ ]:
from IPython.display import HTML, display

def picture(grid, title):
    n = len(grid)
    squares = []
    for y in range(n):
        for x in range(n):
            red = round(255 * grid[y][x] / 100)
            squares.append(f'<rect x="{x*16}" y="{y*16}" width="16" height="16" fill="rgb({red},30,{255-red})"/>')
    return f'<div style="display:inline-block;margin:12px">{title}<br><svg width="{n*16}" height="{n*16}">{"".join(squares)}</svg></div>'

grid = [[0.0]*12 for _ in range(12)]
grid[0] = [100.0]*12
views = [picture(grid, 'Initial plate')]
for step in range(40):
    next_grid = [row[:] for row in grid]
    for y in range(1, 11):
        for x in range(1, 11):
            next_grid[y][x] = (grid[y-1][x]+grid[y+1][x]+grid[y][x-1]+grid[y][x+1])/4
    grid = next_grid
    if step in (0, 9, 39): views.append(picture(grid, f'After {step+1} steps'))
display(HTML(''.join(views)))

## 2. Log in and create a working folder

Replace `YOUR_USERNAME` with your account. The login node is for editing and submitting work. We will request compute resources before running the solver.

```bash
ssh YOUR_USERNAME@ssh.iesci.tech
mkdir -p ~/hpc-labs/m5s1-simple
cd ~/hpc-labs/m5s1-simple
pwd
hostname
module avail GCC
module avail OpenMPI
module load OpenMPI/4.1.5-GCC-12.3.0
module list
which gcc
which mpicc
```

These module names were available on SciTech during preparation. `module list` records what your session actually uses. A module adjusts paths and libraries in your shell; it does not install software in your lab folder.

### Checkpoint 1

Check your folder and compiler paths. Write the hostname and loaded module in `observations.txt`. Explain why a compiler on your laptop does not tell us which compiler the cluster will use.

## 3. Read the solver

In `heat.c`, first find the two arrays, the hot-edge initialisation, the time-step loop and the four-neighbour update. Those lines are the numerical method.

The blocks starting with `#ifdef` select a version at compile time. Initially, ignore the OpenMP and OpenACC blocks. `-DNAIVE` selects a version that copies the new grid back after every step. Without it, we alternate the arrays and avoid the copy. Both calculate the same temperatures.

We store a square grid in a flat array. Cell `(y, x)` is at `y*n+x`. Moving up or down changes the index by `n`; moving left or right changes it by one.

### `heat.c`

In your SSH terminal, paste this block from the lab folder. It creates the file. Read the comments before compiling it.

```bash
cat > heat.c <<'END_FILE'
/* The same stencil runs in serial, with OpenMP, or with OpenACC. */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#ifdef _OPENMP
#include <omp.h>
#endif
#ifdef _OPENACC
#include <openacc.h>
#endif

double seconds(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + 1e-9 * t.tv_nsec;
}

int main(int argc, char **argv) {
    if (argc != 4) {
        fprintf(stderr, "Usage: %s N STEPS output.bin\n", argv[0]);
        return 1;
    }
    int n = atoi(argv[1]), steps = atoi(argv[2]);
    if (n < 3 || n > 4096 || steps < 1 || steps > 100000) return 1;
    size_t count = (size_t)n * n;
    double *a = calloc(count, sizeof(double));
    double *b = calloc(count, sizeof(double));
    if (!a || !b) return 1;
    /* Both arrays have the same fixed edges. The top includes its corners. */
    for (int x = 0; x < n; x++) a[x] = b[x] = 100.0;
    int threads = 1;
    const char *mode = "serial";
#ifdef SERIAL_O0
    mode = "serial_O0";
#endif
#ifdef NAIVE
    mode = "naive";
#endif
#ifdef _OPENMP
    mode = "openmp";
    #pragma omp parallel
    {
        #pragma omp single
        threads = omp_get_num_threads();
    }
#endif
#ifdef _OPENACC
    mode = "openacc";
    /* Refuse a silent CPU fallback in the GPU exercise. */
    if (acc_get_num_devices(acc_device_nvidia) < 1) return 2;
    acc_set_device_type(acc_device_nvidia);
    /* Transfer once; keep both time levels on the GPU for all steps. */
    #pragma acc data copy(a[0:count], b[0:count])
#endif
    {
        double start = seconds();
        for (int step = 0; step < steps; step++) {
#ifdef NAIVE
            double *old = a, *next = b;
#else
            /* Alternate the arrays instead of copying the whole grid. */
            double *old = (step % 2 == 0) ? a : b;
            double *next = (step % 2 == 0) ? b : a;
#endif
#ifdef _OPENMP
            #pragma omp parallel for collapse(2) schedule(static)
#endif
#ifdef _OPENACC
            #pragma acc parallel loop collapse(2) present(old[0:count], next[0:count])
#endif
            for (int y = 1; y < n - 1; y++) {
                for (int x = 1; x < n - 1; x++) {
                    size_t k = (size_t)y * n + x;
                    next[k] = 0.25 * (old[k-n] + old[k+n] + old[k-1] + old[k+1]);
                }
            }
#ifdef NAIVE
            /* This extra full-grid copy is the first cost we will remove. */
            for (size_t k = 0; k < count; k++) a[k] = b[k];
#endif
        }
        /* This timer excludes allocation, file output and GPU data transfers. */
        printf("%s,%d,%d,1,%d,%.9f\n", mode, n, steps, threads, seconds()-start);
    }
#ifdef NAIVE
    double *final = a;
#else
    double *final = (steps % 2 == 0) ? a : b;
#endif
    FILE *file = fopen(argv[3], "wb");
    if (!file) return 1;
    size_t written = fwrite(final, sizeof(double), count, file);
    int closed = fclose(file);
    free(a); free(b);
    return written == count && closed == 0 ? 0 : 1;
}
END_FILE
```

### Compile and run on one allocated core

From the login node, request a small interactive allocation. Wait until Slurm starts the shell, then compile and run there.

```bash
srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:05:00 --pty bash
cd ~/hpc-labs/m5s1-simple
module load OpenMPI/4.1.5-GCC-12.3.0
gcc -O0 -DNAIVE heat.c -o naive
./naive 8 1 first-step.bin
```

The output is one CSV row: mode, grid size, steps, ranks, threads and update-loop seconds. The `.bin` file contains `N*N` double-precision temperatures in row order. Timing excludes file output.

### Checkpoint 2

Run this in the same SSH shell. This checks a known answer rather than comparing two implementations that could share the same mistake.

```bash
python3 - <<'PY'
from array import array
from pathlib import Path
v = array('d'); v.frombytes(Path('first-step.bin').read_bytes())
assert len(v) == 64
assert list(v[:8]) == [100.0]*8
assert list(v[9:15]) == [25.0]*6
assert v[8] == v[15] == 0.0
assert all(x == 0.0 for x in v[16:])
print('PASS: the first step has the expected temperatures')
PY
```

If it fails, stop and inspect the initialisation and indices. Use `exit` to release the interactive allocation when finished.

## 4. Remove copying, then add threads

There are three separate changes to investigate. Removing the copy saves memory traffic. Compiler optimisation changes the generated machine code. OpenMP shares independent cell updates between threads in one process.

Request two cores and compile the same file four ways.

```bash
srun -p cpu -N1 -n1 -c2 --mem=1G --time=00:05:00 --pty bash
cd ~/hpc-labs/m5s1-simple
module load OpenMPI/4.1.5-GCC-12.3.0
gcc -O0 -DNAIVE heat.c -o naive
gcc -O0 -DSERIAL_O0 heat.c -o serial_O0
gcc -O2 heat.c -o serial
gcc -O2 -fopenmp heat.c -o openmp
./naive 128 150 naive.bin
./serial_O0 128 150 serial_O0.bin
./serial 128 150 serial.bin
OMP_NUM_THREADS=2 ./openmp 128 150 openmp.bin
```

OpenMP's `parallel for` divides the iterations. Its implicit barrier finishes the update before the next time step begins. The old array is read-only during each update; each iteration writes a different new cell.

### Checkpoint 3

```bash
python3 - <<'PY'
from array import array
from pathlib import Path
def read(name):
    v = array('d'); v.frombytes(Path(name).read_bytes()); return v
reference = read('serial.bin')
for name in ['naive.bin', 'serial_O0.bin', 'openmp.bin']:
    values = read(name)
    assert len(values) == len(reference)
    error = max(abs(a-b) for a,b in zip(reference, values))
    assert error <= 1e-10
    print(name, 'PASS, maximum error =', error)
PY
```

Compare naive with `serial_O0` to study copying, then `serial_O0` with serial to study compiler optimisation. Compare serial with OpenMP to study threading. A small grid may be slower with threads because starting and synchronising workers costs time.

Record your explanation, then `exit` to release the allocation.

## 5. Divide the grid with MPI

MPI processes have separate arrays. Each rank owns some interior rows and two extra halo rows. Before an update, it exchanges boundary rows with its neighbours. `MPI_Sendrecv` combines a send and receive without requiring every rank to send first.

This example uses equal row blocks: `N-2` must be divisible by the number of ranks. With `N=128` and two ranks, each owns 63 interior rows. Rank 0 keeps the hot physical edge in its upper halo; there is no process above it.

Look for the two halo exchanges, the same stencil, and the final gather. Communication happens once per step; gathering the complete plate happens only at the end.

### `heat_mpi.c`

In your SSH terminal, paste this block from the lab folder. It creates the file. Read the comments before compiling it.

```bash
cat > heat_mpi.c <<'END_FILE'
/* MPI owns rows; OpenMP can share the rows within each MPI process. */
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#ifdef _OPENMP
#include <omp.h>
#endif

int main(int argc, char **argv) {
    int rank, ranks, provided;
    MPI_Init_thread(&argc, &argv, MPI_THREAD_FUNNELED, &provided);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &ranks);
    if (argc != 4 || provided < MPI_THREAD_FUNNELED) MPI_Abort(MPI_COMM_WORLD, 1);
    int n = atoi(argv[1]), steps = atoi(argv[2]);
    /* Equal row blocks keep this teaching example short. */
    if (n < 3 || n > 4096 || steps < 1 || steps > 100000 || (n-2) % ranks != 0 || ranks > n-2) {
        if (rank == 0) fprintf(stderr, "Need N>=3, STEPS>=1 and N-2 divisible by the rank count.\n");
        MPI_Abort(MPI_COMM_WORLD, 1);
    }
    int rows = (n-2) / ranks;
    size_t count = (size_t)(rows+2) * n;
    double *old = calloc(count, sizeof(double));
    double *next = calloc(count, sizeof(double));
    if (!old || !next) MPI_Abort(MPI_COMM_WORLD, 1);
    /* Row 0 and row rows+1 are halos. Rank 0's upper halo is the hot edge. */
    if (rank == 0) for (int x = 0; x < n; x++) old[x] = next[x] = 100.0;
    int up = rank == 0 ? MPI_PROC_NULL : rank-1;
    int down = rank == ranks-1 ? MPI_PROC_NULL : rank+1;
    int threads = 1;
    const char *mode = "mpi";
#ifdef _OPENMP
    mode = "hybrid";
    #pragma omp parallel
    {
        #pragma omp single
        threads = omp_get_num_threads();
    }
#endif
    fprintf(stderr, "rank %d/%d owns %d interior rows, threads=%d\n", rank, ranks, rows, threads);
    MPI_Barrier(MPI_COMM_WORLD);
    double start = MPI_Wtime();
    for (int step = 0; step < steps; step++) {
        /* Send our first row up; receive the lower neighbour into our bottom halo. */
        MPI_Sendrecv(old+n, n, MPI_DOUBLE, up, 0,
                     old+(size_t)(rows+1)*n, n, MPI_DOUBLE, down, 0,
                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        /* Send our last row down; receive the upper neighbour into our top halo. */
        MPI_Sendrecv(old+(size_t)rows*n, n, MPI_DOUBLE, down, 1,
                     old, n, MPI_DOUBLE, up, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
#ifdef _OPENMP
        #pragma omp parallel for collapse(2) schedule(static)
#endif
        for (int y = 1; y <= rows; y++) {
            for (int x = 1; x < n-1; x++) {
                size_t k = (size_t)y*n+x;
                next[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
            }
        }
        double *temporary = old; old = next; next = temporary;
    }
    /* Report the slowest rank, since all ranks are needed to finish the step. */
    double elapsed = MPI_Wtime()-start, maximum;
    MPI_Reduce(&elapsed, &maximum, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);
    double *global = NULL;
    if (rank == 0) {
        global = calloc((size_t)n*n, sizeof(double));
        if (!global) MPI_Abort(MPI_COMM_WORLD, 1);
        for (int x = 0; x < n; x++) global[x] = 100.0;
    }
    /* Gather only owned rows; the root keeps the physical top/bottom edges. */
    MPI_Gather(old+n, rows*n, MPI_DOUBLE, rank == 0 ? global+n : NULL,
               rows*n, MPI_DOUBLE, 0, MPI_COMM_WORLD);
    if (rank == 0) {
        FILE *file = fopen(argv[3], "wb");
        if (!file) MPI_Abort(MPI_COMM_WORLD, 1);
        size_t written = fwrite(global, sizeof(double), (size_t)n*n, file);
        int closed = fclose(file);
        if (written != (size_t)n*n || closed != 0) MPI_Abort(MPI_COMM_WORLD, 1);
        printf("%s,%d,%d,%d,%d,%.9f\n", mode, n, steps, ranks, threads, maximum);
    }
    free(old); free(next); free(global);
    MPI_Finalize();
    return 0;
}
END_FILE
```

### Run MPI, then combine it with OpenMP

Request two processes with two CPU cores per process. This reserves four cores for the hybrid case.

```bash
salloc -p cpu -N1 -n2 -c2 --mem=2G --time=00:05:00
cd ~/hpc-labs/m5s1-simple
module load OpenMPI/4.1.5-GCC-12.3.0
mpicc -O2 heat_mpi.c -o mpi
mpicc -O2 -fopenmp heat_mpi.c -o hybrid
OMP_NUM_THREADS=1 srun -n2 -c2 ./mpi 128 150 mpi.bin
OMP_NUM_THREADS=2 srun -n2 -c2 ./hybrid 128 150 hybrid.bin
```

`salloc` reserves resources and opens an allocation shell; the solver runs on the allocated node when `srun` starts it. `mpicc` is a compiler wrapper that adds the MPI headers and libraries. The MPI-only run uses one active thread per rank; the hybrid run uses two. Both fit this allocation.

### Checkpoint 4

Each run must print two ownership lines, with `/2` in each rank label. The hybrid lines must report two threads. Repeat the full-field comparison from Checkpoint 3 with `mpi.bin` and `hybrid.bin`.

Explain which data the threads share, which data ranks exchange, and why no thread calls MPI inside the parallel loop. The hybrid run has four workers; this is not an equal-core comparison with the two-worker OpenMP run.

If the ownership lines show `/1`, the launcher did not create the intended communicator. Stop and check the MPI module/launcher before timing. Use `exit` to release the allocation.

## 6. Submit one batch job and save the measurements

Instead of typing every run, put the commands in one job file. The `#SBATCH` lines request resources; the remaining lines are ordinary shell commands. There are no build, submission or measurement helper scripts.

The job below repeats every CPU mode three times and saves `timings.csv`. It also runs the one-step check, profiles the naive program if `gprof` is available, and attempts ParaView after the CPU work. Each operation is visible in this file.

### `view.py`

In your SSH terminal, paste this block from the lab folder. It creates the file. Read the comments before compiling it.

```bash
cat > view.py <<'END_FILE'
"""Use ParaView's Python, not the notebook's Python, for this file."""
from array import array
from pathlib import Path
import sys
from paraview.simple import (OpenDataFile, GetActiveViewOrCreate, Show, ColorBy,
                            GetColorTransferFunction, SaveScreenshot)

n = int(sys.argv[1])
values = array('d')
values.frombytes(Path(sys.argv[2]).read_bytes())
assert len(values) == n*n
# VTK adds grid geometry and a scalar name to the solver's raw numbers.
with open('temperature.vtk', 'w') as file:
    file.write(f'# vtk DataFile Version 3.0\nHeat plate, Celsius\nASCII\nDATASET STRUCTURED_POINTS\nDIMENSIONS {n} {n} 1\nORIGIN 0 0 0\nSPACING 1 1 1\nPOINT_DATA {n*n}\nSCALARS Temperature double 1\nLOOKUP_TABLE default\n')
    file.write('\n'.join(map(str, values)))
field = OpenDataFile(str(Path('temperature.vtk').resolve()))
field.UpdatePipeline()
assert field.GetDataInformation().GetNumberOfPoints() == n*n
assert field.PointData['Temperature'].GetRange(0) == (0.0, 100.0)
view = GetActiveViewOrCreate('RenderView')
image = Show(field, view)
ColorBy(image, ('POINTS', 'Temperature'))
GetColorTransferFunction('Temperature').RescaleTransferFunction(0, 100)
image.SetScalarBarVisibility(view, True)
view.CameraParallelProjection = 1
view.CameraPosition = [(n-1)/2, (n-1)/2, 2*n]
view.CameraFocalPoint = [(n-1)/2, (n-1)/2, 0]
view.CameraViewUp = [0, -1, 0]
view.CameraParallelScale = n*0.6
# Unlike simple.Render, StillRender does not create an X-window interactor.
view.StillRender()
SaveScreenshot('paraview.png', view, ImageResolution=[900, 700])
print('PARAVIEW_PASS')
END_FILE
```

`view.py` uses ParaView's Python environment. The VTK header adds grid dimensions and a scalar name to the raw numbers. We fix the colour scale at 0 to 100 °C so images from different runs are comparable.

The job uses `pvbatch` and Mesa software rendering because its CPU allocation has no display. A rendering failure is recorded in the log. The notebook also has a lightweight preview, which is not a ParaView screenshot.

### `run.slurm`

In your SSH terminal, paste this block from the lab folder. It creates the file. Read the comments before compiling it.

```bash
cat > run.slurm <<'END_FILE'
#!/bin/bash -l
#SBATCH --job-name=m5s1-simple
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=2
#SBATCH --cpus-per-task=2
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=cpu-%j.out
set -eu
cd "$SLURM_SUBMIT_DIR"
N=${N:-128}; STEPS=${STEPS:-150}; REPEATS=${REPEATS:-3}
module load OpenMPI/4.1.5-GCC-12.3.0
module list 2>modules.txt
gcc --version | head -1
# Each compiler command makes one change that we can explain.
gcc -O0 -DNAIVE heat.c -o naive
gcc -O0 -DSERIAL_O0 heat.c -o serial_O0
gcc -O2 heat.c -o serial
gcc -O2 -fopenmp heat.c -o openmp
mpicc -O2 heat_mpi.c -o mpi
mpicc -O2 -fopenmp heat_mpi.c -o hybrid
export OMP_DYNAMIC=FALSE OMP_PLACES=cores OMP_PROC_BIND=close
printf 'mode,n,steps,ranks,threads,seconds\n' > timings.csv
for repeat in $(seq 1 "$REPEATS"); do
    # One log row and a separate field file per measured run.
    srun -n1 -c1 ./naive "$N" "$STEPS" "naive-$repeat.bin" >>timings.csv
    srun -n1 -c1 ./serial_O0 "$N" "$STEPS" "serial_O0-$repeat.bin" >>timings.csv
    srun -n1 -c1 ./serial "$N" "$STEPS" "serial-$repeat.bin" >>timings.csv
    OMP_NUM_THREADS=2 srun -n1 -c2 ./openmp "$N" "$STEPS" "openmp-$repeat.bin" >>timings.csv
    OMP_NUM_THREADS=1 srun -n2 -c2 ./mpi "$N" "$STEPS" "mpi-$repeat.bin" >>timings.csv
    OMP_NUM_THREADS=2 srun -n2 -c2 ./hybrid "$N" "$STEPS" "hybrid-$repeat.bin" >>timings.csv
done
# A one-step run has a known answer, independent of the other implementations.
srun -n1 -c1 ./serial 8 1 first-step.bin > first-step.csv
# Profiling is separate from the timing comparison, because instrumentation has a cost.
if command -v gprof >/dev/null; then
    gcc -O0 -g -pg -DNAIVE heat.c -o heat_profile
    srun -n1 -c1 ./heat_profile 512 600 profile.bin > profile-time.csv
    gprof ./heat_profile gmon.out > profile.txt
fi
echo CPU_DONE
# Visualisation is post-processing. A render failure does not erase CPU results.
if module load ParaView/5.11.2-foss-2023a; then
    module list 2>paraview-modules.txt
    export LIBGL_ALWAYS_SOFTWARE=true GALLIUM_DRIVER=llvmpipe EGL_PLATFORM=surfaceless LP_NUM_THREADS=1
    unset DISPLAY WAYLAND_DISPLAY PV_DEBUG_REMOTE_RENDERING
    ulimit -c 0
    if pvbatch --force-offscreen-rendering view.py "$N" serial-1.bin; then
        echo PARAVIEW_DONE
    else
        echo 'PARAVIEW_FAILED: inspect this log for the rendering error.'
    fi
else
    echo 'PARAVIEW_UNAVAILABLE: the module could not be loaded.'
fi
END_FILE
```

### Submit and inspect

From the login node, in the lab folder:

```bash
sbatch run.slurm
squeue -u "$USER"
```

Write down the returned job ID. Replace `JOBID` below with that number.

```bash
sacct -j JOBID --format=JobID,State,ExitCode
cat cpu-JOBID.out
cat timings.csv
cat modules.txt
ls -lh *.bin temperature.vtk paraview.png
```

### Checkpoint 5

Slurm must report `COMPLETED`, and the log must contain `CPU_DONE`. With three repeats and six CPU modes, `timings.csv` must have 18 data rows. Compare every field with `serial-1.bin` using Checkpoint 3's code. Check the one-step field too.

ParaView success is separate: look for `PARAVIEW_DONE` and a valid `paraview.png`. A missing PNG is not evidence of a wrong heat solver. Read the rendering error in the log.

Use the median of the three loop times for each mode. Serial/OpenMP measure the stencil loop; MPI/hybrid include halo communication. Neither includes process startup or file output. Record job duration separately if discussing the whole application.

## 7. Read a profile

The batch job builds a separate instrumented naive executable with `-pg`. If `gprof` is installed, it saves `profile.txt`.

```bash
head -40 profile.txt
```

Find `main` and the sampling information. The stencil and copy loops are both inside `main`, so this profile cannot assign a separate cost to each line. Our naive versus `serial_O0` comparison helps isolate the copy's effect. Short runs may produce too few samples; absence from the table does not prove that an operation was free.

Why do we keep the instrumented executable out of the main timing comparison?

## 8. Try the GPU with OpenACC

Return to `heat.c`. The `_OPENACC` blocks keep both arrays on the GPU across all steps. The `parallel loop` directive describes independent cell updates. Data transfers happen around the time-step loop, rather than once per step.

We use the same source and boundary conditions. The code refuses to silently fall back to a CPU when no NVIDIA device is available.

This is an extension after the CPU checkpoints. GPU jobs can remain queued when all devices are busy.

### `gpu.slurm`

In your SSH terminal, paste this block from the lab folder. It creates the file. Read the comments before compiling it.

```bash
cat > gpu.slurm <<'END_FILE'
#!/bin/bash -l
#SBATCH --job-name=m5s1-openacc
#SBATCH --partition=gpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --gres=gpu:1
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=gpu-%j.out
set -eu
cd "$SLURM_SUBMIT_DIR"
module load nvhpc/25.7
module list 2>gpu-modules.txt
nvidia-smi
# Compile the same source. The _OPENACC blocks become active in this build.
nvc -O2 -acc -gpu=ccnative -Minfo=accel heat.c -o heat_acc
./heat_acc "${N:-128}" "${STEPS:-150}" openacc.bin > gpu-time.csv
echo GPU_DONE
END_FILE
```

From the login node:

```bash
module avail nvhpc
sbatch gpu.slurm
squeue -u "$USER"
```

After completion, read `gpu-JOBID.out` and `gpu-time.csv`. Compare `openacc.bin` with `serial-1.bin` using the full-field check. A queued job has not run; successful compilation alone is not a successful GPU calculation.

### Checkpoint 6

Look for compiler offload information, the allocated device in `nvidia-smi`, `GPU_DONE`, and a field error no larger than `1e-10`. The GPU timer excludes transfers, so it measures something narrower than complete application time. Explain why a tiny plate is a poor GPU speed benchmark.

The [extended notebook](M5S1_hybrid_computing_extended.ipynb) keeps the longer CUDA implementation, library compatibility exercise and additional examples for later study. They are not prerequisites for this practical.

## 9. Make the experiment larger

Start only after the fields agree. Predict what will happen before submitting each case.

```bash
cp timings.csv timings-128-150.csv
N=512 STEPS=150 REPEATS=3 sbatch run.slurm
```

Wait for completion, check the fields, and copy the measurements before submitting the next case:

```bash
cp timings.csv timings-512-150.csv
N=512 STEPS=600 REPEATS=3 sbatch run.slurm
```

The batch job overwrites filenames in its working folder, so run these cases one at a time and save each table before the next starts. Keep the allocation and compiler settings fixed. If you also rerun the GPU, pass the same `N` and `STEPS` to `gpu.slurm`.

At fixed steps, increasing `N` from 128 to 512 increases interior-cell work by about 16 times. Increasing steps from 150 to 600 multiplies work by four. Larger loops can make worker overhead less dominant, but speedup is not guaranteed.

Here grid spacing stays at one, so increasing `N` makes the physical domain larger. It is not refinement of the same plate or a strong-scaling experiment.

Hand in your tables, one temperature image, compiler/module information, and short answers:

1. Which change helped most: removing copying, compiler optimisation, threading or MPI?
2. Which comparisons used the same number of active workers?
3. Why must halo exchange finish before updating cells beside a rank boundary?
4. Which time excludes startup, transfers or I/O?
5. What changed when the workload grew? Support your explanation with measurements.

---
## Teaching demonstration

This final section reproduces the practical from Jupyter. Run All saves the same files, submits the CPU and OpenACC jobs, waits, checks the fields and displays the available results. It needs a connected SciTech kernel with `sbatch` on its path. No benchmark runs on the notebook host.

The next cell stores the same source shown above. It is collapsed to keep the controls short. A saved job ID prevents resubmitting the same demonstration when you rerun the cells. This uses its own folder, separate from students' SSH work.

In [ ]:
FILES = {'heat.c': '/* The same stencil runs in serial, with OpenMP, or with OpenACC. */\n#include <stdio.h>\n#include <stdlib.h>\n#include <time.h>\n#ifdef _OPENMP\n#include <omp.h>\n#endif\n#ifdef _OPENACC\n#include <openacc.h>\n#endif\n\ndouble seconds(void) {\n    struct timespec t;\n    clock_gettime(CLOCK_MONOTONIC, &t);\n    return t.tv_sec + 1e-9 * t.tv_nsec;\n}\n\nint main(int argc, char **argv) {\n    if (argc != 4) {\n        fprintf(stderr, "Usage: %s N STEPS output.bin\\n", argv[0]);\n        return 1;\n    }\n    int n = atoi(argv[1]), steps = atoi(argv[2]);\n    if (n < 3 || n > 4096 || steps < 1 || steps > 100000) return 1;\n    size_t count = (size_t)n * n;\n    double *a = calloc(count, sizeof(double));\n    double *b = calloc(count, sizeof(double));\n    if (!a || !b) return 1;\n    /* Both arrays have the same fixed edges. The top includes its corners. */\n    for (int x = 0; x < n; x++) a[x] = b[x] = 100.0;\n    int threads = 1;\n    const char *mode = "serial";\n#ifdef SERIAL_O0\n    mode = "serial_O0";\n#endif\n#ifdef NAIVE\n    mode = "naive";\n#endif\n#ifdef _OPENMP\n    mode = "openmp";\n    #pragma omp parallel\n    {\n        #pragma omp single\n        threads = omp_get_num_threads();\n    }\n#endif\n#ifdef _OPENACC\n    mode = "openacc";\n    /* Refuse a silent CPU fallback in the GPU exercise. */\n    if (acc_get_num_devices(acc_device_nvidia) < 1) return 2;\n    acc_set_device_type(acc_device_nvidia);\n    /* Transfer once; keep both time levels on the GPU for all steps. */\n    #pragma acc data copy(a[0:count], b[0:count])\n#endif\n    {\n        double start = seconds();\n        for (int step = 0; step < steps; step++) {\n#ifdef NAIVE\n            double *old = a, *next = b;\n#else\n            /* Alternate the arrays instead of copying the whole grid. */\n            double *old = (step % 2 == 0) ? a : b;\n            double *next = (step % 2 == 0) ? b : a;\n#endif\n#ifdef _OPENMP\n            #pragma omp parallel for collapse(2) schedule(static)\n#endif\n#ifdef _OPENACC\n            #pragma acc parallel loop collapse(2) present(old[0:count], next[0:count])\n#endif\n            for (int y = 1; y < n - 1; y++) {\n                for (int x = 1; x < n - 1; x++) {\n                    size_t k = (size_t)y * n + x;\n                    next[k] = 0.25 * (old[k-n] + old[k+n] + old[k-1] + old[k+1]);\n                }\n            }\n#ifdef NAIVE\n            /* This extra full-grid copy is the first cost we will remove. */\n            for (size_t k = 0; k < count; k++) a[k] = b[k];\n#endif\n        }\n        /* This timer excludes allocation, file output and GPU data transfers. */\n        printf("%s,%d,%d,1,%d,%.9f\\n", mode, n, steps, threads, seconds()-start);\n    }\n#ifdef NAIVE\n    double *final = a;\n#else\n    double *final = (steps % 2 == 0) ? a : b;\n#endif\n    FILE *file = fopen(argv[3], "wb");\n    if (!file) return 1;\n    size_t written = fwrite(final, sizeof(double), count, file);\n    int closed = fclose(file);\n    free(a); free(b);\n    return written == count && closed == 0 ? 0 : 1;\n}\n', 'heat_mpi.c': '/* MPI owns rows; OpenMP can share the rows within each MPI process. */\n#include <mpi.h>\n#include <stdio.h>\n#include <stdlib.h>\n#ifdef _OPENMP\n#include <omp.h>\n#endif\n\nint main(int argc, char **argv) {\n    int rank, ranks, provided;\n    MPI_Init_thread(&argc, &argv, MPI_THREAD_FUNNELED, &provided);\n    MPI_Comm_rank(MPI_COMM_WORLD, &rank);\n    MPI_Comm_size(MPI_COMM_WORLD, &ranks);\n    if (argc != 4 || provided < MPI_THREAD_FUNNELED) MPI_Abort(MPI_COMM_WORLD, 1);\n    int n = atoi(argv[1]), steps = atoi(argv[2]);\n    /* Equal row blocks keep this teaching example short. */\n    if (n < 3 || n > 4096 || steps < 1 || steps > 100000 || (n-2) % ranks != 0 || ranks > n-2) {\n        if (rank == 0) fprintf(stderr, "Need N>=3, STEPS>=1 and N-2 divisible by the rank count.\\n");\n        MPI_Abort(MPI_COMM_WORLD, 1);\n    }\n    int rows = (n-2) / ranks;\n    size_t count = (size_t)(rows+2) * n;\n    double *old = calloc(count, sizeof(double));\n    double *next = calloc(count, sizeof(double));\n    if (!old || !next) MPI_Abort(MPI_COMM_WORLD, 1);\n    /* Row 0 and row rows+1 are halos. Rank 0\'s upper halo is the hot edge. */\n    if (rank == 0) for (int x = 0; x < n; x++) old[x] = next[x] = 100.0;\n    int up = rank == 0 ? MPI_PROC_NULL : rank-1;\n    int down = rank == ranks-1 ? MPI_PROC_NULL : rank+1;\n    int threads = 1;\n    const char *mode = "mpi";\n#ifdef _OPENMP\n    mode = "hybrid";\n    #pragma omp parallel\n    {\n        #pragma omp single\n        threads = omp_get_num_threads();\n    }\n#endif\n    fprintf(stderr, "rank %d/%d owns %d interior rows, threads=%d\\n", rank, ranks, rows, threads);\n    MPI_Barrier(MPI_COMM_WORLD);\n    double start = MPI_Wtime();\n    for (int step = 0; step < steps; step++) {\n        /* Send our first row up; receive the lower neighbour into our bottom halo. */\n        MPI_Sendrecv(old+n, n, MPI_DOUBLE, up, 0,\n                     old+(size_t)(rows+1)*n, n, MPI_DOUBLE, down, 0,\n                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n        /* Send our last row down; receive the upper neighbour into our top halo. */\n        MPI_Sendrecv(old+(size_t)rows*n, n, MPI_DOUBLE, down, 1,\n                     old, n, MPI_DOUBLE, up, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n#ifdef _OPENMP\n        #pragma omp parallel for collapse(2) schedule(static)\n#endif\n        for (int y = 1; y <= rows; y++) {\n            for (int x = 1; x < n-1; x++) {\n                size_t k = (size_t)y*n+x;\n                next[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);\n            }\n        }\n        double *temporary = old; old = next; next = temporary;\n    }\n    /* Report the slowest rank, since all ranks are needed to finish the step. */\n    double elapsed = MPI_Wtime()-start, maximum;\n    MPI_Reduce(&elapsed, &maximum, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);\n    double *global = NULL;\n    if (rank == 0) {\n        global = calloc((size_t)n*n, sizeof(double));\n        if (!global) MPI_Abort(MPI_COMM_WORLD, 1);\n        for (int x = 0; x < n; x++) global[x] = 100.0;\n    }\n    /* Gather only owned rows; the root keeps the physical top/bottom edges. */\n    MPI_Gather(old+n, rows*n, MPI_DOUBLE, rank == 0 ? global+n : NULL,\n               rows*n, MPI_DOUBLE, 0, MPI_COMM_WORLD);\n    if (rank == 0) {\n        FILE *file = fopen(argv[3], "wb");\n        if (!file) MPI_Abort(MPI_COMM_WORLD, 1);\n        size_t written = fwrite(global, sizeof(double), (size_t)n*n, file);\n        int closed = fclose(file);\n        if (written != (size_t)n*n || closed != 0) MPI_Abort(MPI_COMM_WORLD, 1);\n        printf("%s,%d,%d,%d,%d,%.9f\\n", mode, n, steps, ranks, threads, maximum);\n    }\n    free(old); free(next); free(global);\n    MPI_Finalize();\n    return 0;\n}\n', 'view.py': '"""Use ParaView\'s Python, not the notebook\'s Python, for this file."""\nfrom array import array\nfrom pathlib import Path\nimport sys\nfrom paraview.simple import (OpenDataFile, GetActiveViewOrCreate, Show, ColorBy,\n                            GetColorTransferFunction, SaveScreenshot)\n\nn = int(sys.argv[1])\nvalues = array(\'d\')\nvalues.frombytes(Path(sys.argv[2]).read_bytes())\nassert len(values) == n*n\n# VTK adds grid geometry and a scalar name to the solver\'s raw numbers.\nwith open(\'temperature.vtk\', \'w\') as file:\n    file.write(f\'# vtk DataFile Version 3.0\\nHeat plate, Celsius\\nASCII\\nDATASET STRUCTURED_POINTS\\nDIMENSIONS {n} {n} 1\\nORIGIN 0 0 0\\nSPACING 1 1 1\\nPOINT_DATA {n*n}\\nSCALARS Temperature double 1\\nLOOKUP_TABLE default\\n\')\n    file.write(\'\\n\'.join(map(str, values)))\nfield = OpenDataFile(str(Path(\'temperature.vtk\').resolve()))\nfield.UpdatePipeline()\nassert field.GetDataInformation().GetNumberOfPoints() == n*n\nassert field.PointData[\'Temperature\'].GetRange(0) == (0.0, 100.0)\nview = GetActiveViewOrCreate(\'RenderView\')\nimage = Show(field, view)\nColorBy(image, (\'POINTS\', \'Temperature\'))\nGetColorTransferFunction(\'Temperature\').RescaleTransferFunction(0, 100)\nimage.SetScalarBarVisibility(view, True)\nview.CameraParallelProjection = 1\nview.CameraPosition = [(n-1)/2, (n-1)/2, 2*n]\nview.CameraFocalPoint = [(n-1)/2, (n-1)/2, 0]\nview.CameraViewUp = [0, -1, 0]\nview.CameraParallelScale = n*0.6\n# Unlike simple.Render, StillRender does not create an X-window interactor.\nview.StillRender()\nSaveScreenshot(\'paraview.png\', view, ImageResolution=[900, 700])\nprint(\'PARAVIEW_PASS\')\n', 'run.slurm': '#!/bin/bash -l\n#SBATCH --job-name=m5s1-simple\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=2\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=cpu-%j.out\nset -eu\ncd "$SLURM_SUBMIT_DIR"\nN=${N:-128}; STEPS=${STEPS:-150}; REPEATS=${REPEATS:-3}\nmodule load OpenMPI/4.1.5-GCC-12.3.0\nmodule list 2>modules.txt\ngcc --version | head -1\n# Each compiler command makes one change that we can explain.\ngcc -O0 -DNAIVE heat.c -o naive\ngcc -O0 -DSERIAL_O0 heat.c -o serial_O0\ngcc -O2 heat.c -o serial\ngcc -O2 -fopenmp heat.c -o openmp\nmpicc -O2 heat_mpi.c -o mpi\nmpicc -O2 -fopenmp heat_mpi.c -o hybrid\nexport OMP_DYNAMIC=FALSE OMP_PLACES=cores OMP_PROC_BIND=close\nprintf \'mode,n,steps,ranks,threads,seconds\\n\' > timings.csv\nfor repeat in $(seq 1 "$REPEATS"); do\n    # One log row and a separate field file per measured run.\n    srun -n1 -c1 ./naive "$N" "$STEPS" "naive-$repeat.bin" >>timings.csv\n    srun -n1 -c1 ./serial_O0 "$N" "$STEPS" "serial_O0-$repeat.bin" >>timings.csv\n    srun -n1 -c1 ./serial "$N" "$STEPS" "serial-$repeat.bin" >>timings.csv\n    OMP_NUM_THREADS=2 srun -n1 -c2 ./openmp "$N" "$STEPS" "openmp-$repeat.bin" >>timings.csv\n    OMP_NUM_THREADS=1 srun -n2 -c2 ./mpi "$N" "$STEPS" "mpi-$repeat.bin" >>timings.csv\n    OMP_NUM_THREADS=2 srun -n2 -c2 ./hybrid "$N" "$STEPS" "hybrid-$repeat.bin" >>timings.csv\ndone\n# A one-step run has a known answer, independent of the other implementations.\nsrun -n1 -c1 ./serial 8 1 first-step.bin > first-step.csv\n# Profiling is separate from the timing comparison, because instrumentation has a cost.\nif command -v gprof >/dev/null; then\n    gcc -O0 -g -pg -DNAIVE heat.c -o heat_profile\n    srun -n1 -c1 ./heat_profile 512 600 profile.bin > profile-time.csv\n    gprof ./heat_profile gmon.out > profile.txt\nfi\necho CPU_DONE\n# Visualisation is post-processing. A render failure does not erase CPU results.\nif module load ParaView/5.11.2-foss-2023a; then\n    module list 2>paraview-modules.txt\n    export LIBGL_ALWAYS_SOFTWARE=true GALLIUM_DRIVER=llvmpipe EGL_PLATFORM=surfaceless LP_NUM_THREADS=1\n    unset DISPLAY WAYLAND_DISPLAY PV_DEBUG_REMOTE_RENDERING\n    ulimit -c 0\n    if pvbatch --force-offscreen-rendering view.py "$N" serial-1.bin; then\n        echo PARAVIEW_DONE\n    else\n        echo \'PARAVIEW_FAILED: inspect this log for the rendering error.\'\n    fi\nelse\n    echo \'PARAVIEW_UNAVAILABLE: the module could not be loaded.\'\nfi\n', 'gpu.slurm': '#!/bin/bash -l\n#SBATCH --job-name=m5s1-openacc\n#SBATCH --partition=gpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=1\n#SBATCH --gres=gpu:1\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=gpu-%j.out\nset -eu\ncd "$SLURM_SUBMIT_DIR"\nmodule load nvhpc/25.7\nmodule list 2>gpu-modules.txt\nnvidia-smi\n# Compile the same source. The _OPENACC blocks become active in this build.\nnvc -O2 -acc -gpu=ccnative -Minfo=accel heat.c -o heat_acc\n./heat_acc "${N:-128}" "${STEPS:-150}" openacc.bin > gpu-time.csv\necho GPU_DONE\n'}

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, time
from array import array
from IPython.display import display, HTML, Image

N, STEPS, REPEATS = 128, 150, 3
RUN_GPU = True
WAIT_SECONDS = 1800
settings = dict(N=str(N), STEPS=str(STEPS), REPEATS=str(REPEATS))
fingerprint = hashlib.sha256(json.dumps([FILES, settings], sort_keys=True).encode()).hexdigest()[:10]
folder = Path.home() / 'hpc-labs' / ('m5s1-simple-demo-' + fingerprint)
folder.mkdir(parents=True, exist_ok=True)
for name, source in FILES.items(): (folder/name).write_text(source)
print('Demonstration folder:', folder, flush=True)
print('Python:', sys.executable, flush=True)

def submit_and_wait(kind, script):
    import shutil
    if not shutil.which('sbatch'):
        print('NOT RUN: sbatch is unavailable in this kernel.', flush=True)
        return False
    marker = folder / (kind + '-job.txt')
    if marker.exists():
        job = marker.read_text().strip()
    else:
        env = os.environ.copy()
        for key in ('SLURM_MEM_PER_CPU', 'SLURM_MEM_PER_NODE', 'SLURM_MEM_PER_GPU'):
            env.pop(key, None)
        env.update(settings)
        # Save intent before submission; an uncertain response must not cause a duplicate.
        marker.write_text('SUBMITTING')
        result = subprocess.run(['sbatch', '--parsable', script], cwd=folder, env=env,
                                capture_output=True, text=True, timeout=35)
        if result.returncode: raise RuntimeError(result.stderr)
        job = result.stdout.strip().split(';')[0]
        if not job.isdigit(): raise RuntimeError('Unexpected sbatch response: ' + result.stdout)
        marker.write_text(job)
    if not job.isdigit():
        print('Submission needs checking in Slurm; no duplicate was submitted.', flush=True)
        return False
    print(kind, 'job', job, flush=True)
    deadline = time.monotonic() + WAIT_SECONDS
    previous = None
    last_notice = 0
    while time.monotonic() < deadline:
        query = subprocess.run(['sacct','-n','-P','-X','-j',job,'-o','State,ExitCode'],
                               capture_output=True, text=True, timeout=25)
        state = query.stdout.strip().splitlines()
        status = state[0].split('|')[0].strip().rstrip('+') if state else 'WAITING FOR ACCOUNTING'
        if status != previous or time.monotonic()-last_notice >= 30:
            print(kind, status, flush=True); previous = status; last_notice = time.monotonic()
        if status == 'COMPLETED': return True
        if status.split()[0] in ('FAILED','CANCELLED','TIMEOUT','OUT_OF_MEMORY','NODE_FAIL','PREEMPTED'):
            print((folder/f'{kind}-{job}.out').read_text(errors='replace'), flush=True)
            return False
        time.sleep(5)
    print(kind, 'still unfinished. Rerun this cell to resume waiting for the same job.', flush=True)
    return False

cpu_ok = submit_and_wait('cpu', 'run.slurm')

### Check the CPU results and measurements

We compare complete fields, not just a checksum. The independent first-step check is small enough to calculate by hand. Timing values below come from the submitted job.

In [ ]:
import csv, math, statistics

def read_field(name, n):
    values = array('d')
    values.frombytes((folder/name).read_bytes())
    assert len(values) == n*n and all(math.isfinite(x) for x in values)
    return values

if cpu_ok:
    first = read_field('first-step.bin', 8)
    assert list(first[:8]) == [100.0]*8
    assert list(first[8:16]) == [0.0]+[25.0]*6+[0.0]
    assert all(v == 0.0 for v in first[16:])
    reference = read_field('serial-1.bin', N)
    assert list(reference[:N]) == [100.0]*N
    assert all(reference[y*N] == reference[y*N+N-1] == 0 for y in range(1, N))
    assert all(v == 0 for v in reference[-N:])
    assert min(reference) >= 0 and max(reference) <= 100
    rows = list(csv.DictReader((folder/'timings.csv').open()))
    modes = ['naive','serial_O0','serial','openmp','mpi','hybrid']
    assert len(rows) == len(modes)*REPEATS
    for mode in modes:
        selected = [row for row in rows if row['mode'] == mode]
        assert len(selected) == REPEATS
        assert all(int(row['n']) == N and int(row['steps']) == STEPS for row in selected)
        expected_ranks, expected_threads = {'openmp':(1,2), 'mpi':(2,1), 'hybrid':(2,2)}.get(mode,(1,1))
        assert all((int(row['ranks']),int(row['threads'])) == (expected_ranks,expected_threads) for row in selected)
        values = [float(row['seconds']) for row in selected]
        assert all(math.isfinite(t) and t >= 0 for t in values)
        for repeat in range(1, REPEATS+1):
            field = read_field(f'{mode}-{repeat}.bin', N)
            error = max(abs(a-b) for a,b in zip(reference, field))
            assert error <= 1e-10, (mode, error)
        print(f'{mode:12s} PASS   median loop = {statistics.median(values):.6f} s')
    print('PASS independent first-step check and all complete CPU fields')
    print('Profiling:', 'profile.txt saved' if (folder/'profile.txt').exists() else 'gprof unavailable')
    if (folder/'profile.txt').exists(): print((folder/'profile.txt').read_text()[:2500])
else:
    print('CPU results are not complete; inspect the job status above.')

### Inspect the field and GPU result

The quick preview uses the actual serial output and standard Python. The separate PNG is displayed only when ParaView has produced it successfully. The GPU field is checked against the same CPU reference.

In [ ]:
if cpu_ok:
    # Sample only for display; the validation above compares every cell.
    stride = max(1, N//64)
    indices = list(range(0, N, stride))
    if indices[-1] != N-1: indices.append(N-1)
    grid = [[reference[y*N+x] for x in indices] for y in indices]
    display(HTML(picture(grid, 'Calculated serial field: 0 to 100 °C')))

gpu_ok = submit_and_wait('gpu', 'gpu.slurm') if RUN_GPU else False
if gpu_ok and cpu_ok:
    gpu_field = read_field('openacc.bin', N)
    error = max(abs(a-b) for a,b in zip(reference, gpu_field))
    assert error <= 1e-10, error
    print('OpenACC PASS, maximum field error =', error)
    print((folder/'gpu-time.csv').read_text())
elif not RUN_GPU:
    print('GPU demonstration disabled.')
else:
    print('OpenACC has not passed. Check the job status and log above.')

# Display the actual ParaView screenshot last, after checking its completion marker.
if cpu_ok:
    job = (folder/'cpu-job.txt').read_text().strip()
    log = (folder/f'cpu-{job}.out').read_text(errors='replace')
    png = folder/'paraview.png'
    if 'PARAVIEW_DONE' in log and png.exists() and png.read_bytes().startswith(b'\x89PNG\r\n\x1a\n'):
        print('ParaView image from the completed batch job')
        display(Image(filename=str(png)))
    else:
        print('No verified ParaView image. CPU results are separate from rendering status.')
        print(log[-3500:])